# 02 · Frenet Target — 잔차를 운동정렬 축에서 학습 (EXP #14, LB 0.6878)

직전 base = v3(LB 0.6678). 잔차를 Cartesian(x/y/z) 대신 **운동정렬(Frenet) 축**에서 예측한다.

**가정**: 모기 운동은 진행방향(L)·구심(N, 곡률)·평면이탈(B)로 자연 분해된다. 절대좌표는 챔버 비대칭·드리프트와 얽혀 분포 의존이 크지만, **feature와 target을 같은 motion-aligned 축에 정렬**하면 학습 신호가 강해진다(잔차 std 비대칭 Cart 1:0.96:0.68 → Frenet 1:0.81:0.62).

**실험**: residual을 (res_L, res_N, res_B)로 투영해 LGBM 3개가 각 성분을 예측. feature·target 정렬 정도가 다른 5 변형(V1 미스매치 ~ V3 완전정합)을 Phase 1 sanity(G_S5) → Phase 2(W1) 3-gate로 비교한다.

**결과**: 전 변형 3-gate 통과, V2(38 feat) 채택 → **LB 0.6878 (+0.0200)**. 단일 lever 최대 효과 — D 라인 전체(W1+D4+D5) 합보다 큰 단일 step. LB-OOF gap도 +0.0092 확대 = train→private generalization 개선.

## 셀 0 — imports + 상수

In [ ]:
import os, time, json
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold

DT, DT_PRED = 0.04, 0.08
SEED, N_FOLDS = 42, 5
MINORITY_THRESHOLD = 5.0
HIT_RADIUS = 0.01

# D4 채택 (v3 동일)
BEST_SIGMA = 0.0035
BEST_MAX_W = 2.5

# G_S5_T threshold (D1 Cart no-weight reference)
G_S5_T_THR = 0.6430

# 3-gate (plan §11.9.2, vs V0 reference)
G1_THR = 0.0017       # Bonferroni-lite (single seed std 0.001 × √n≈3)
G2_TOL = 0.002
G3_TOL = 0.005

os.makedirs('results', exist_ok=True)
print(f'D4 채택: σ={BEST_SIGMA*1000}mm, max_w={BEST_MAX_W}')
print(f'G_S5_T threshold: {G_S5_T_THR} (D1 Cart no-weight reference)')
print(f'3-gate (Phase 2): G1>+{G1_THR}, G2 major>-{G2_TOL}, G3 minor>-{G3_TOL}')

## 셀 1 — helper (v3 동일: physics_baseline, hit_rate, w_gaussian, splits)

In [ ]:
def _norm(x):
    return np.linalg.norm(x, axis=-1)


def physics_baseline(traj):
    """CV 2-point diff (v3와 비트 단위 동일)."""
    p0, p_m40 = traj[:, -1, :], traj[:, -2, :]
    v_last = (p0 - p_m40) / DT
    return (p0 + v_last * DT_PRED).astype(np.float32)


def hit_rate(pred, label, r=HIT_RADIUS):
    return float((np.linalg.norm(pred - label, axis=1) < r).mean())


def w_gaussian(e, sigma, max_w, r=HIT_RADIUS):
    """D4 W1 weight (v3 동일)."""
    return np.clip(1.0 + (max_w - 1.0) * np.exp(-(e - r)**2 / (2 * sigma**2)),
                   0.5, max_w)


def make_splits_full(minority_int, seed=SEED):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    return [(tr, va) for tr, va in skf.split(np.arange(len(minority_int)), minority_int)]


lgbm_params = dict(
    objective='regression_l1', metric='mae',
    learning_rate=0.05, num_leaves=31, min_data_in_leaf=5,
    max_bin=511, n_estimators=500, random_state=SEED, verbose=-1,
)
print('helper 정의 완료')

## 셀 2 — Drive mount + zip 해제 (캐시 우선)

In [ ]:
CACHE_TRAJ_TR = 'traj_train.npy'
CACHE_Y_TR    = 'y_train.npy'
CACHE_TRAJ_TS = 'traj_test.npy'

DATA_DIR = None
if not (os.path.exists(CACHE_TRAJ_TR) and os.path.exists(CACHE_Y_TR) and os.path.exists(CACHE_TRAJ_TS)):
    from google.colab import drive
    drive.mount('/content/drive')
    ZIP_PATH = '/content/drive/MyDrive/open.zip'
    !unzip -q -o "{ZIP_PATH}" -d /content/
    DATA_DIR = '/content/open' if os.path.exists('/content/open/sample_submission.csv') else '/content'
    print(f'DATA_DIR = {DATA_DIR}')
else:
    print('모든 캐시 존재 — Drive mount 생략')

## 셀 3 — train 로딩 (캐시 우선)

In [ ]:
import pandas as pd

if os.path.exists(CACHE_TRAJ_TR) and os.path.exists(CACHE_Y_TR):
    traj_train = np.load(CACHE_TRAJ_TR)
    y_train    = np.load(CACHE_Y_TR)
    print(f'train cache: traj {traj_train.shape}, y {y_train.shape}')
else:
    labels = pd.read_csv(f'{DATA_DIR}/train_labels.csv')
    train_ids = labels['id'].tolist()
    y_train = labels[['x','y','z']].values.astype(np.float32)
    traj_train = np.empty((len(train_ids), 11, 3), dtype=np.float32)
    for i, tid in enumerate(train_ids):
        df = pd.read_csv(f'{DATA_DIR}/train/{tid}.csv')
        traj_train[i] = df[['x','y','z']].values
    np.save(CACHE_TRAJ_TR, traj_train)
    np.save(CACHE_Y_TR, y_train)
    print(f'train fresh: traj {traj_train.shape}, y {y_train.shape}')

assert traj_train.shape == (10000, 11, 3)
assert y_train.shape == (10000, 3)
assert np.isfinite(traj_train).all() and np.isfinite(y_train).all()

## 셀 4 — test 로딩 (sample_submission 순서 보존)

In [ ]:
if DATA_DIR is None:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')
    if not os.path.exists('/content/open/sample_submission.csv') and not os.path.exists('/content/sample_submission.csv'):
        ZIP_PATH = '/content/drive/MyDrive/open.zip'
        !unzip -q -o "{ZIP_PATH}" -d /content/
    DATA_DIR = '/content/open' if os.path.exists('/content/open/sample_submission.csv') else '/content'

sample_sub = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')
test_ids = sample_sub['id'].tolist()
assert len(test_ids) == 10000

if os.path.exists(CACHE_TRAJ_TS):
    traj_test = np.load(CACHE_TRAJ_TS)
    print(f'test cache: traj {traj_test.shape}')
else:
    traj_test = np.empty((len(test_ids), 11, 3), dtype=np.float32)
    for i, tid in enumerate(test_ids):
        df = pd.read_csv(f'{DATA_DIR}/test/{tid}.csv')
        traj_test[i] = df[['x','y','z']].values
    np.save(CACHE_TRAJ_TS, traj_test)
    print(f'test fresh: traj {traj_test.shape}')

assert traj_test.shape == (10000, 11, 3)
assert np.isfinite(traj_test).all()

## 셀 5 — physics base + residual vector (Cartesian intermediate)

residual = y - base는 basis-independent 3D vector. 여기선 Cart basis로 계산해두고, 셀 8에서 Frenet basis로 회전해 LGBM target으로 사용.

In [ ]:
base_train = physics_baseline(traj_train)
base_test  = physics_baseline(traj_test)
residual_cart_train = (y_train - base_train).astype(np.float32)        # intermediate, Cart basis

assert residual_cart_train.shape == (10000, 3)
assert np.isfinite(residual_cart_train).all()

print(f'physics HR (train): {hit_rate(base_train, y_train):.4f}  (목표 ≈ 0.5788)')
print(f'residual_cart std (mm): x={residual_cart_train[:,0].std()*1000:.2f}  '
      f'y={residual_cart_train[:,1].std()*1000:.2f}  z={residual_cart_train[:,2].std()*1000:.2f}')

## 셀 6 — Frenet frame 구축 (train + test)

frame: `eL = v_last/||v_last||`, `eN = (a_sm − (a_sm·eL)eL)/||...||`, `eB = eL × eN`. a_sm = w·a[-3:] (v3 동일 정의).

fallback (||a_perp||<1e-6): z-up cross, 2차 fallback y-up. eda_frenet.py 측정 시 hard fallback 0회 (majority의 ||a_sm||<1은 38.2%지만 정확히 0은 없음).

In [ ]:
def build_frenet_batch(v_last, a_sm, fb_thresh=1e-6):
    """v_last, a_sm: (N, 3) → eL, eN, eB: each (N, 3). Vectorized."""
    vn = np.linalg.norm(v_last, axis=1, keepdims=True)
    eL = v_last / (vn + 1e-9)
    a_perp = a_sm - (a_sm * eL).sum(1, keepdims=True) * eL
    apn = np.linalg.norm(a_perp, axis=1, keepdims=True)
    eN_primary = a_perp / (apn + 1e-9)
    z_up = np.array([0., 0., 1.]); y_up = np.array([0., 1., 0.])
    eN_fb1 = np.cross(eL, z_up); n1 = np.linalg.norm(eN_fb1, axis=1, keepdims=True)
    eN_fb2 = np.cross(eL, y_up); n2 = np.linalg.norm(eN_fb2, axis=1, keepdims=True)
    eN_fb  = np.where(n1 < 1e-6, eN_fb2 / (n2 + 1e-9), eN_fb1 / (n1 + 1e-9))
    use_fb = (apn < fb_thresh)
    eN = np.where(use_fb, eN_fb, eN_primary)
    eB = np.cross(eL, eN)
    eB = eB / (np.linalg.norm(eB, axis=1, keepdims=True) + 1e-9)
    return eL.astype(np.float32), eN.astype(np.float32), eB.astype(np.float32), use_fb.squeeze(-1)


def compute_kinematics(traj):
    """traj (N,11,3) → v_last, a_last, jerk_last, a_sm (모두 (N,3))"""
    v = (traj[:, 1:, :] - traj[:, :-1, :]) / DT
    a = (v[:, 1:, :] - v[:, :-1, :]) / DT
    v_last = v[:, -1, :]
    a_last = a[:, -1, :]
    jerk_last = (a[:, -1, :] - a[:, -2, :]) / DT
    w = np.array([1, 2, 3]) / 6
    a_sm = (a[:, -3:, :] * w[None, :, None]).sum(axis=1)
    return v_last.astype(np.float32), a_last.astype(np.float32), jerk_last.astype(np.float32), a_sm.astype(np.float32)


vl_tr, al_tr, jl_tr, asm_tr = compute_kinematics(traj_train)
vl_ts, al_ts, jl_ts, asm_ts = compute_kinematics(traj_test)

eL_tr, eN_tr, eB_tr, fb_tr = build_frenet_batch(vl_tr, asm_tr)
eL_ts, eN_ts, eB_ts, fb_ts = build_frenet_batch(vl_ts, asm_ts)

# frame sanity (inverse round-trip)
rL = (residual_cart_train * eL_tr).sum(1)
rN = (residual_cart_train * eN_tr).sum(1)
rB = (residual_cart_train * eB_tr).sum(1)
rec = rL[:, None]*eL_tr + rN[:, None]*eN_tr + rB[:, None]*eB_tr
inv_err = np.abs(rec - residual_cart_train).max()
print(f'frame inverse round-trip max abs err: {inv_err:.2e}  (목표 < 1e-5)')
assert inv_err < 1e-5, f'inverse 정확도 fail: {inv_err}'
print(f'hard fallback (||a_perp||<1e-6) — train: {fb_tr.sum()}, test: {fb_ts.sum()}')

# orthogonality sanity
ortho_LN = np.abs((eL_tr * eN_tr).sum(1)).max()
ortho_LB = np.abs((eL_tr * eB_tr).sum(1)).max()
ortho_NB = np.abs((eN_tr * eB_tr).sum(1)).max()
print(f'orthogonality max |eL·eN|={ortho_LN:.2e}, |eL·eB|={ortho_LB:.2e}, |eN·eB|={ortho_NB:.2e}')

## 셀 7 — 5 변형 feature builder

v3 31-feature + 7 Frenet projection을 미리 다 계산해두고, 변형별로 column slicing.

- **V1** (31): v3 31
- **V2** (38): v3 31 + 7 Frenet
- **V3** (26): v3 19 scalars + 7 Frenet
- **V4** (13): EXP #13 Phase B lean (A 8 + vy_std + distance_r + va_dot + a_tang_last + speed_diff_half)
- **V5** (20): V4 + 7 Frenet

In [ ]:
FEAT_V3_31 = [
    'vx_last', 'vy_last', 'vz_last', 'ax_last', 'ay_last', 'az_last',
    'speed_last', 'acc_norm_last',                                          # 0-7   A (8)
    'ax_w3', 'ay_w3', 'az_w3', 'acc_norm_w3',                               # 8-11  B (4)
    'vx_std', 'vy_std', 'vz_std', 'ax_std', 'ay_std', 'az_std', 'path_eff', # 12-18 C (7)
    'distance_r', 'radial_v',                                               # 19-20 D (2)
    'turn_mean', 'cos_turn_last', 'va_dot',                                 # 21-23 E (3)
    'jerk_x_last', 'jerk_y_last', 'jerk_z_last',                            # 24-26 F (3)
    'a_tang_last', 'a_cent_last',                                           # 27-28 G (2)
    'speed_diff_half', 'turn_mean_half_diff',                               # 29-30 H (2)
]
assert len(FEAT_V3_31) == 31

FEAT_FRENET_7 = ['a_N', 'a_B', 'j_L', 'j_N', 'j_B', 'aw_L', 'aw_N']
# 제외: v_L=speed_last 중복, v_N/v_B 구조적 0, a_L=a_tang_last 중복, aw_B 구조적 0

# v3 19 scalars = 31 − 12 Cart dirs (vx/vy/vz/ax/ay/az_last + ax/ay/az_w3 + jerk × 3)
CART_DIRS_12 = ['vx_last','vy_last','vz_last','ax_last','ay_last','az_last',
                'ax_w3','ay_w3','az_w3','jerk_x_last','jerk_y_last','jerk_z_last']
FEAT_V3_19_SCALARS = [f for f in FEAT_V3_31 if f not in CART_DIRS_12]
assert len(FEAT_V3_19_SCALARS) == 19

# V4 (EXP #13 Phase B lean): A 8 + vy_std + distance_r + va_dot + a_tang_last + speed_diff_half
FEAT_V4_13 = ['vx_last','vy_last','vz_last','ax_last','ay_last','az_last',
              'speed_last','acc_norm_last',
              'vy_std','distance_r','va_dot','a_tang_last','speed_diff_half']
assert len(FEAT_V4_13) == 13

FEAT_V1 = FEAT_V3_31[:]                                    # 31
FEAT_V2 = FEAT_V3_31 + FEAT_FRENET_7                       # 38
FEAT_V3 = FEAT_V3_19_SCALARS + FEAT_FRENET_7               # 26
FEAT_V4 = FEAT_V4_13[:]                                    # 13
FEAT_V5 = FEAT_V4_13 + FEAT_FRENET_7                       # 20

VARIANTS = {'V1': FEAT_V1, 'V2': FEAT_V2, 'V3': FEAT_V3, 'V4': FEAT_V4, 'V5': FEAT_V5}
for n, f in VARIANTS.items():
    print(f'  {n}: {len(f)} feat')

### 셀 7-b — v3 31 + Frenet 7 = 38-column 통합 build (per train/test)

In [ ]:
def build_features_38(traj, eL, eN, eB):
    """v3 31 + Frenet 7 = 38 dim. column order = FEAT_V3_31 + FEAT_FRENET_7."""
    p = traj
    v = (p[:, 1:, :] - p[:, :-1, :]) / DT
    a = (v[:, 1:, :] - v[:, :-1, :]) / DT
    v_last, a_last = v[:, -1, :], a[:, -1, :]
    speed_last, acc_norm_last = _norm(v_last), _norm(a_last)
    w = np.array([1, 2, 3]) / 6
    a3 = a[:, -3:, :]
    a_w3 = (a3 * w[None, :, None]).sum(axis=1)
    acc_norm_w3 = _norm(a_w3)
    v_std, a_std = v.std(axis=1), a.std(axis=1)
    seg = _norm(p[:, 1:, :] - p[:, :-1, :])
    path_eff = _norm(p[:, -1, :] - p[:, 0, :]) / (seg.sum(1) + 1e-9)
    p0 = p[:, -1, :]
    distance_r = _norm(p0)
    radial_v = (p0 * v_last).sum(1) / (distance_r + 1e-9)
    v_n = v / (_norm(v)[..., None] + 1e-9)
    cos_consec = (v_n[:, 1:, :] * v_n[:, :-1, :]).sum(-1).clip(-1, 1)
    turn = np.arccos(cos_consec)
    turn_mean, cos_turn_last = turn.mean(1), cos_consec[:, -1]
    va_dot = (v_last * a_last).sum(1) / (speed_last * acc_norm_last + 1e-9)
    jerk_last = (a[:, -1, :] - a[:, -2, :]) / DT
    v_a_dot = (v_last * a_last).sum(1)
    v_cross_a = np.cross(v_last, a_last)
    a_tang_last = v_a_dot / (speed_last + 1e-9)
    a_cent_last = _norm(v_cross_a) / (speed_last + 1e-9)
    speed_seq = _norm(v)
    speed_diff_half = speed_seq[:, 5:].mean(1) - speed_seq[:, :5].mean(1)
    turn_mean_half_diff = turn[:, 4:].mean(1) - turn[:, :4].mean(1)

    # v3 31 column block
    v3_31 = np.column_stack([
        v_last, a_last, speed_last, acc_norm_last,            # A 8
        a_w3, acc_norm_w3,                                    # B 4
        v_std, a_std, path_eff,                               # C 7
        distance_r, radial_v,                                 # D 2
        turn_mean, cos_turn_last, va_dot,                     # E 3
        jerk_last,                                            # F 3
        a_tang_last, a_cent_last,                             # G 2
        speed_diff_half, turn_mean_half_diff,                 # H 2
    ]).astype(np.float32)
    assert v3_31.shape[1] == 31

    # Frenet 7 projection block (a_N, a_B, j_L, j_N, j_B, aw_L, aw_N)
    a_N  = (a_last * eN).sum(1)
    a_B  = (a_last * eB).sum(1)
    j_L  = (jerk_last * eL).sum(1)
    j_N  = (jerk_last * eN).sum(1)
    j_B  = (jerk_last * eB).sum(1)
    aw_L = (a_w3 * eL).sum(1)
    aw_N = (a_w3 * eN).sum(1)
    fren_7 = np.column_stack([a_N, a_B, j_L, j_N, j_B, aw_L, aw_N]).astype(np.float32)
    assert fren_7.shape[1] == 7

    return np.column_stack([v3_31, fren_7])  # (N, 38)


X38_train = build_features_38(traj_train, eL_tr, eN_tr, eB_tr)
X38_test  = build_features_38(traj_test,  eL_ts, eN_ts, eB_ts)
assert X38_train.shape == (10000, 38) and X38_test.shape == (10000, 38)
assert np.isfinite(X38_train).all() and np.isfinite(X38_test).all()

ALL_FEATS = FEAT_V3_31 + FEAT_FRENET_7
FEAT_IDX = {name: i for i, name in enumerate(ALL_FEATS)}

def select_variant(X38, variant_feats):
    idx = [FEAT_IDX[f] for f in variant_feats]
    return X38[:, idx]

for n, f in VARIANTS.items():
    Xv = select_variant(X38_train, f)
    print(f'  {n}: train {Xv.shape}, test {select_variant(X38_test, f).shape}')

## 셀 8 — Frenet target 변환 (실제 LGBM target)

residual_cart → (res_L, res_N, res_B) projection. LGBM 3 model이 각 Frenet 성분을 예측.

In [ ]:
r_L = (residual_cart_train * eL_tr).sum(1)
r_N = (residual_cart_train * eN_tr).sum(1)
r_B = (residual_cart_train * eB_tr).sum(1)
residual_fren_train = np.stack([r_L, r_N, r_B], axis=1).astype(np.float32)

assert residual_fren_train.shape == (10000, 3)
assert np.isfinite(residual_fren_train).all()

print('residual std (mm)')
print(f'  Cart x={residual_cart_train[:,0].std()*1000:.2f}  y={residual_cart_train[:,1].std()*1000:.2f}  z={residual_cart_train[:,2].std()*1000:.2f}')
print(f'  Fren L={r_L.std()*1000:.2f}  N={r_N.std()*1000:.2f}  B={r_B.std()*1000:.2f}')
print(f'  Fren ratio L:N:B = 1 : {r_N.std()/r_L.std():.2f} : {r_B.std()/r_L.std():.2f}')

## 셀 9 — 5-fold split (full 10000, stratify=minority)

In [ ]:
acc_norm_last_tr = _norm(al_tr)
minority_mask_tr = acc_norm_last_tr >= MINORITY_THRESHOLD
print(f'minority(acc_norm_last >= 5): {minority_mask_tr.sum()}/{len(minority_mask_tr)} ({100*minority_mask_tr.mean():.2f}%)')

folds = make_splits_full(minority_mask_tr.astype(int), seed=SEED)
for k, (tr_in, val_in) in enumerate(folds):
    print(f'fold {k}: tr {len(tr_in)}, val {len(val_in)}, val_minority {minority_mask_tr[val_in].mean():.3f}')

## 셀 10 — Phase 1 (no-weight, G_S5_T sanity)

5 변형 × 5 fold × 3 axis = 75 model. Frenet target 학습 → inverse transform으로 Cart residual 복원 → HR 측정.

**G_S5_T**: HR ≥ 0.6430 (D1 Cart no-weight reference). fail 시 Phase 2 skip.

In [ ]:
def inverse_transform_batch(resid_fren, eL, eN, eB):
    """resid_fren (N,3), eL/eN/eB (N,3) → resid_cart (N,3)"""
    return (resid_fren[:, 0:1] * eL +
            resid_fren[:, 1:2] * eN +
            resid_fren[:, 2:3] * eB)


phase1_results = {}
phase1_oof = {}   # variant → oof_resid_fren (10000, 3)

t0 = time.time()
for vname, feats in VARIANTS.items():
    Xv_tr = select_variant(X38_train, feats)
    oof_resid_fren = np.full((10000, 3), np.nan, dtype=np.float32)
    for tr_in, val_in in folds:
        for ax in range(3):
            m = lgb.LGBMRegressor(**lgbm_params)
            m.fit(Xv_tr[tr_in], residual_fren_train[tr_in, ax],
                  eval_set=[(Xv_tr[val_in], residual_fren_train[val_in, ax])],
                  callbacks=[lgb.early_stopping(50, verbose=False)])
            oof_resid_fren[val_in, ax] = m.predict(Xv_tr[val_in]).astype(np.float32)
    assert np.isfinite(oof_resid_fren).all()

    # inverse transform + HR
    oof_resid_cart = inverse_transform_batch(oof_resid_fren, eL_tr, eN_tr, eB_tr)
    pred_oof = base_train + oof_resid_cart
    oof_e = np.linalg.norm(y_train - pred_oof, axis=1).astype(np.float32)

    HR_overall  = hit_rate(pred_oof, y_train)
    HR_major    = hit_rate(pred_oof[~minority_mask_tr], y_train[~minority_mask_tr])
    HR_minor    = hit_rate(pred_oof[ minority_mask_tr], y_train[ minority_mask_tr])
    g_s5_t      = HR_overall >= G_S5_T_THR

    phase1_results[vname] = dict(
        n_feat=len(feats),
        HR_overall=HR_overall, HR_major=HR_major, HR_minor=HR_minor,
        resid_mean_mm=float(oof_e.mean()*1000),
        G_S5_T=g_s5_t,
        oof_e=oof_e,
    )
    phase1_oof[vname] = oof_resid_fren
    print(f'  {vname} ({len(feats):2d} feat): overall={HR_overall:.4f}  '
          f'major={HR_major:.4f}  minor={HR_minor:.4f}  '
          f'||resid||={oof_e.mean()*1000:.2f}mm  '
          f'G_S5_T={"O" if g_s5_t else "X"} (>={G_S5_T_THR})')

print(f'\nPhase 1 done in {time.time()-t0:.1f}s')
np.save('results/t_phase1_oof_v1.npy', phase1_oof['V1'])

## 셀 11 — Phase 1 결과 표 + G_S5_T 통과 변형 list

In [ ]:
print(f'\n{"변형":<5} {"n_feat":>6}  {"overall":>8}  {"major":>8}  {"minor":>8}  {"resid(mm)":>10}  {"G_S5_T":>7}')
print('-' * 60)
for vname, r in phase1_results.items():
    flag = 'O' if r['G_S5_T'] else 'X'
    print(f'{vname:<5} {r["n_feat"]:>6}  {r["HR_overall"]:>8.4f}  '
          f'{r["HR_major"]:>8.4f}  {r["HR_minor"]:>8.4f}  '
          f'{r["resid_mean_mm"]:>10.2f}  {flag:>7}')

phase2_pass = [v for v, r in phase1_results.items() if r['G_S5_T']]
print(f'\nPhase 2 진입 변형: {phase2_pass} ({len(phase2_pass)}/5)')

if len(phase2_pass) == 0:
    print('\n⚠️  Phase 1 5/5 fail — T 라인 폐기 후보. plan §11.9.3 backup (Cart target 5 변형) 검토.')

## 셀 12 — Phase 2 (D4 W1, σ=3.5mm max_w=2.5)

**V0** (Cart feat × Cart target): 3-gate reference 측정 (v3 D4 W1 full 10k Phase 2 재현).

**Vi** (G_S5_T 통과): Frenet target W1 학습 + test 예측 누적.

변형마다 OOF residual norm이 다르므로 W1 weight도 변형별 산출 (plan §11.7.1).

In [ ]:
# ── V0 — Cart × Cart D4 W1 reference (3-gate baseline) ──
# Phase 1 V1과 동일 31 feat이지만 target이 Cart x/y/z
print('=== V0 (Cart × Cart D4 W1 reference) — Phase 1 no-weight 우선 ===')
Xv0_tr = select_variant(X38_train, FEAT_V3_31)
Xv0_ts = select_variant(X38_test,  FEAT_V3_31)

oof_resid_cart_v0_p1 = np.full((10000, 3), np.nan, dtype=np.float32)
t0 = time.time()
for tr_in, val_in in folds:
    for ax in range(3):
        m = lgb.LGBMRegressor(**lgbm_params)
        m.fit(Xv0_tr[tr_in], residual_cart_train[tr_in, ax],
              eval_set=[(Xv0_tr[val_in], residual_cart_train[val_in, ax])],
              callbacks=[lgb.early_stopping(50, verbose=False)])
        oof_resid_cart_v0_p1[val_in, ax] = m.predict(Xv0_tr[val_in]).astype(np.float32)
pred_v0_p1 = base_train + oof_resid_cart_v0_p1
v0_p1_HR_overall = hit_rate(pred_v0_p1, y_train)
oof_e_v0 = np.linalg.norm(y_train - pred_v0_p1, axis=1).astype(np.float32)
print(f'V0 Phase 1 no-weight: overall={v0_p1_HR_overall:.4f}  (EXP #13 F0=0.6494 참고)')
print(f'Phase 1 V0 done in {time.time()-t0:.1f}s')

print('\n=== V0 — Phase 2 W1 (3-gate reference) ===')
w_v0 = w_gaussian(oof_e_v0, BEST_SIGMA, BEST_MAX_W)
oof_resid_cart_v0_p2 = np.full((10000, 3), np.nan, dtype=np.float32)
pred_test_resid_v0 = np.zeros((10000, 3), dtype=np.float32)
t0 = time.time()
for tr_in, val_in in folds:
    for ax in range(3):
        m = lgb.LGBMRegressor(**lgbm_params)
        m.fit(Xv0_tr[tr_in], residual_cart_train[tr_in, ax],
              sample_weight=w_v0[tr_in],
              eval_set=[(Xv0_tr[val_in], residual_cart_train[val_in, ax])],
              callbacks=[lgb.early_stopping(50, verbose=False)])
        oof_resid_cart_v0_p2[val_in, ax] = m.predict(Xv0_tr[val_in]).astype(np.float32)
        pred_test_resid_v0[:, ax] += m.predict(Xv0_ts).astype(np.float32) / N_FOLDS
pred_v0 = base_train + oof_resid_cart_v0_p2
v0_HR_overall = hit_rate(pred_v0, y_train)
v0_HR_major   = hit_rate(pred_v0[~minority_mask_tr], y_train[~minority_mask_tr])
v0_HR_minor   = hit_rate(pred_v0[ minority_mask_tr], y_train[ minority_mask_tr])
print(f'V0 (Cart × Cart D4 W1): overall={v0_HR_overall:.4f}  major={v0_HR_major:.4f}  minor={v0_HR_minor:.4f}')
print(f'  (v3 D4 reference 0.6665 / 0.7359 / 0.2952과 일치 여부 sanity check)')
print(f'Phase 2 V0 done in {time.time()-t0:.1f}s')

v0_ref = dict(overall=v0_HR_overall, major=v0_HR_major, minor=v0_HR_minor)

### 셀 12-b — Vi (G_S5_T 통과 변형) Phase 2 W1 + test 예측

In [ ]:
phase2_results = {'V0': dict(n_feat=31, **v0_ref, test_pred_resid=pred_test_resid_v0)}

if len(phase2_pass) == 0:
    print('Phase 2 skip — G_S5_T 통과 변형 없음')
else:
    t0 = time.time()
    for vname in phase2_pass:
        Xv_tr = select_variant(X38_train, VARIANTS[vname])
        Xv_ts = select_variant(X38_test,  VARIANTS[vname])
        oof_e_v = phase1_results[vname]['oof_e']
        w_v = w_gaussian(oof_e_v, BEST_SIGMA, BEST_MAX_W)

        oof_resid_fren_w1 = np.full((10000, 3), np.nan, dtype=np.float32)
        pred_test_resid_fren = np.zeros((10000, 3), dtype=np.float32)
        for tr_in, val_in in folds:
            for ax in range(3):
                m = lgb.LGBMRegressor(**lgbm_params)
                m.fit(Xv_tr[tr_in], residual_fren_train[tr_in, ax],
                      sample_weight=w_v[tr_in],
                      eval_set=[(Xv_tr[val_in], residual_fren_train[val_in, ax])],
                      callbacks=[lgb.early_stopping(50, verbose=False)])
                oof_resid_fren_w1[val_in, ax] = m.predict(Xv_tr[val_in]).astype(np.float32)
                pred_test_resid_fren[:, ax] += m.predict(Xv_ts).astype(np.float32) / N_FOLDS

        # inverse transform → Cart
        oof_resid_cart_w1 = inverse_transform_batch(oof_resid_fren_w1, eL_tr, eN_tr, eB_tr)
        pred_test_resid_cart = inverse_transform_batch(pred_test_resid_fren, eL_ts, eN_ts, eB_ts)
        pred_v = base_train + oof_resid_cart_w1

        HR_overall = hit_rate(pred_v, y_train)
        HR_major   = hit_rate(pred_v[~minority_mask_tr], y_train[~minority_mask_tr])
        HR_minor   = hit_rate(pred_v[ minority_mask_tr], y_train[ minority_mask_tr])

        phase2_results[vname] = dict(
            n_feat=len(VARIANTS[vname]),
            overall=HR_overall, major=HR_major, minor=HR_minor,
            test_pred_resid=pred_test_resid_cart,
        )
        print(f'  {vname} ({len(VARIANTS[vname]):2d} feat): overall={HR_overall:.4f}  '
              f'major={HR_major:.4f}  minor={HR_minor:.4f}  '
              f'Δoverall vs V0 = {HR_overall - v0_ref["overall"]:+.4f}')
    print(f'\nPhase 2 Vi done in {time.time()-t0:.1f}s')

## 셀 13 — Phase 2 결과 표 + 3-gate (vs V0)

G1: Δoverall > +0.0017, G2: Δmajor > −0.002, G3: Δminor > −0.005. 모두 strict 통과해야 채택.

In [ ]:
print(f'\n{"변형":<5} {"n":>3}  {"overall":>8}  {"Δ overall":>10}  {"major":>8}  {"minor":>8}  {"G1":>3} {"G2":>3} {"G3":>3}  {"pass":>4}')
print('-' * 80)
passed_variants = []
for vname, r in phase2_results.items():
    if vname == 'V0':
        print(f'{vname:<5} {r["n_feat"]:>3}  {r["overall"]:>8.4f}  {"0.0000":>10}  '
              f'{r["major"]:>8.4f}  {r["minor"]:>8.4f}  {"-":>3} {"-":>3} {"-":>3}  {"ref":>4}')
        continue
    d_o = r['overall'] - v0_ref['overall']
    d_M = r['major']   - v0_ref['major']
    d_m = r['minor']   - v0_ref['minor']
    g1 = d_o > G1_THR
    g2 = d_M > -G2_TOL
    g3 = d_m > -G3_TOL
    pass3 = g1 and g2 and g3
    if pass3:
        passed_variants.append(vname)
    f = lambda b: 'O' if b else 'X'
    print(f'{vname:<5} {r["n_feat"]:>3}  {r["overall"]:>8.4f}  {d_o:>+10.4f}  '
          f'{r["major"]:>8.4f}  {r["minor"]:>8.4f}  {f(g1):>3} {f(g2):>3} {f(g3):>3}  {f(pass3):>4}')

print(f'\n3-gate 통과 변형: {passed_variants} ({len(passed_variants)}개)')

if not passed_variants:
    print('\n⚠️  Phase 2 3-gate 0/n 통과 — T 라인 ceiling 확인. plan §11.9.3: polish patch 또는 라인 종료 검토.')
else:
    # 채택: overall HR 최고 변형 (multi-tie 시 minor HR 보조)
    best_v = max(passed_variants, key=lambda v: (phase2_results[v]['overall'], phase2_results[v]['minor']))
    print(f'\n채택: {best_v} (overall={phase2_results[best_v]["overall"]:.4f})')

## 셀 14 — submission 생성 (3-gate 통과 변형 채택 시)

In [ ]:
submission_path = None
if passed_variants:
    best_v = max(passed_variants, key=lambda v: (phase2_results[v]['overall'], phase2_results[v]['minor']))
    pred_test_abs = base_test + phase2_results[best_v]['test_pred_resid']
    assert pred_test_abs.shape == (10000, 3)
    assert np.isfinite(pred_test_abs).all()

    submission_df = pd.DataFrame({
        'id': test_ids,
        'x': pred_test_abs[:, 0],
        'y': pred_test_abs[:, 1],
        'z': pred_test_abs[:, 2],
    })
    submission_path = f'submission_t_{best_v.lower()}.csv'
    submission_df.to_csv(submission_path, index=False)
    print(f'{submission_path} 저장 완료 ({len(submission_df)} rows)')
    print(submission_df.head(3))
    for i, name in enumerate(['x','y','z']):
        a = pred_test_abs[:, i]
        print(f'  {name}: mean {a.mean():.3f}, std {a.std():.3f}, range [{a.min():.3f}, {a.max():.3f}]')
else:
    print('submission 생성 skip — 3-gate 통과 변형 없음')

## 셀 15 — meta 저장 + Drive 복사 + 로컬 다운로드

In [ ]:
# JSON dump
def safe(x):
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.ndarray): return None
    return x

p1_safe = {v: {k: safe(val) for k, val in r.items() if k != 'oof_e'} for v, r in phase1_results.items()}
p2_safe = {v: {k: safe(val) for k, val in r.items() if k != 'test_pred_resid'} for v, r in phase2_results.items()}

meta = {
    'protocol': 'T 라인 (Frenet target ablation) — plan §11',
    'sigma': BEST_SIGMA, 'max_w': BEST_MAX_W,
    'seed': SEED, 'n_folds': N_FOLDS,
    'G_S5_T_threshold': G_S5_T_THR,
    'G1_threshold': G1_THR, 'G2_tol': G2_TOL, 'G3_tol': G3_TOL,
    'minority_threshold': MINORITY_THRESHOLD,
    'frame_inv_err': float(inv_err),
    'phase1_results': p1_safe,
    'phase2_results': p2_safe,
    'phase2_passed': phase2_pass,
    'passed_variants_3gate': passed_variants,
    'v0_reference': {k: float(v) for k, v in v0_ref.items()},
    'submission': submission_path,
}
with open('results/t_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print('results/t_meta.json 저장')

# Drive 복사 + 다운로드 (Colab 환경)
try:
    from google.colab import drive, files
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')

    DRIVE_BASE = '/content/drive/MyDrive'
    results_drive = os.path.join(DRIVE_BASE, 'results')
    os.makedirs(results_drive, exist_ok=True)
    !cp -r results/* {results_drive}/
    print(f'results/* -> {results_drive} 복사 완료')

    if submission_path:
        !cp {submission_path} {DRIVE_BASE}/
        print(f'{submission_path} -> {DRIVE_BASE}/ 복사 완료')
        files.download(submission_path)
    else:
        print('submission 없음 — 다운로드 skip')
except ImportError:
    print('Colab 환경 아님 — Drive 복사 skip')